In [ ]:
import pandas as pd

import sys
sys.path.append('..')
from helpers import (
    get_plant_direct_use,
    get_plant_prime_movers
)

In [80]:
plant_counties = (
    pd.read_csv('../data/power_plant_locations/power_plant_counties.csv')
    .drop_duplicates(subset='Plant Code')
    [['Plant Code', 'FIPS']]
)

In [89]:
county_direct_use_by_year = {}
for load_year in range(2016, 2024):
    plant_direct_use = get_plant_direct_use(load_year)
    plant_prime_movers = get_plant_prime_movers(load_year)
    
    plant_direct_use = (
        plant_direct_use.merge(plant_prime_movers, on='Plant Code', how='left')
        .merge(plant_counties, on='Plant Code', how='left')
        .dropna(subset='FIPS')
    )
    plant_direct_use['direct_use_mwh'] = (
        plant_direct_use['direct_use_mwh'].fillna(0) * plant_direct_use['percent_of_net_generation'].fillna(1)
    )
    plant_direct_use['is_pv'] = plant_direct_use['prime_mover'] == 'PV'
    
    county_direct_use = (
        plant_direct_use.groupby(['FIPS', 'is_pv'], dropna=False)
        ['direct_use_mwh']
        .sum(numeric_only=True)
    )

    county_direct_use_by_year[load_year] = county_direct_use
    print(f"{load_year} done")

2016 done
2017 done
2018 done
2019 done
2020 done
2021 done
2022 done
2023 done


In [91]:
county_direct_use = pd.concat(county_direct_use_by_year, axis=1)

In [94]:
county_direct_use.to_csv('../data/county_direct_use.csv')